# Separable 1D convolutions + SE attention -- AF 4 training

Trains **two new model variants** on OCMR cardiac cine at AF 4, following the paper's
suggested future directions (Sec. 5):

1. **Separable conv** (`--conv-mode separable`): replaces the joint `(3, kernel_t)` 2D
   convolution with sequential `(3,1)` along ky + `(1, kernel_t)` along t. 33% fewer
   parameters per conv layer; processes each spatio-temporal dimension independently.
2. **Separable conv + SE** (`--conv-mode separable --use-se`): adds a squeeze-and-excitation
   channel-attention block after each conv stack. Adaptive per-input feature weighting
   with ~1% extra parameters.

Both are compared against the existing `dyn_af4_kt3_yt_v2` (joint conv, no SE) which
must already be trained. **Requires the preprocessed OCMR data on Drive** from the
original training notebook.

**Run top to bottom.** Each variant takes ~3 h on a T4 (same as the original). Safe to
re-run after disconnects; training resumes from the exact batch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/odls_dynamic"
DRIVE_DATA_DIR = f"{DRIVE_ROOT}/ocmr_preprocessed"
LOCAL_DATA_DIR = "/content/ocmr_preprocessed"
RAW_TMP_DIR = "/content/ocmr_tmp"

REPO_URL = "https://github.com/Shambhawi419/1D_MRI.git"
REPO_DIR = "/content/1D_MRI"
CODE_DIR = f"{REPO_DIR}/odls_dynamic"

AF = 4
SPARSE_DOMAIN = "yt"
EPOCHS = 40
BATCH_SIZE = 32
TIME_LIMIT_MIN = 180

# The two new runs: separable only, then separable + SE.
RUNS = [
    {"name": "dyn_af4_kt3_sep_yt_v2",    "conv_mode": "separable", "use_se": False},
    {"name": "dyn_af4_kt3_sep_se_yt_v2", "conv_mode": "separable", "use_se": True},
]

In [ ]:
import os, subprocess, sys
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
sys.path.insert(0, CODE_DIR)

from google.colab import drive


def ensure_drive():
    if not os.path.ismount('/content/drive'):
        print("Google Drive is not connected -- reconnecting ...")
        drive.mount('/content/drive', force_remount=True)
    if not os.path.isdir(DRIVE_DATA_DIR):
        raise RuntimeError(f"{DRIVE_DATA_DIR} not found -- is this the account that owns odls_dynamic?")


def flush_drive():
    print("flushing Google Drive (waiting for uploads to finish -- don't stop this cell) ...")
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
    print("Drive flushed -- everything is saved.")


!nvidia-smi --query-gpu=name,memory.total --format=csv

## Self-test (1-2 min, no data needed)

In [ ]:
!cd {CODE_DIR} && python -u selftest.py

## Copy preprocessed data to local disk (~1-2 min)

In [ ]:
import shutil, time

ensure_drive()
src_dir, dst_dir = f"{DRIVE_DATA_DIR}/slices", f"{LOCAL_DATA_DIR}/slices"
os.makedirs(dst_dir, exist_ok=True)
for attempt in range(1, 6):
    try:
        names = sorted(n for n in os.listdir(src_dir) if n.endswith(".npy"))
        copied = 0
        for name in names:
            src, dst = f"{src_dir}/{name}", f"{dst_dir}/{name}"
            if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
                continue
            shutil.copy(src, dst + ".tmp")
            os.replace(dst + ".tmp", dst)
            copied += 1
        for name in ("manifest.csv", "split.csv"):
            shutil.copy(f"{DRIVE_DATA_DIR}/{name}", f"{LOCAL_DATA_DIR}/{name}")
        print(f"done: copied {copied} new files, {len(names)} slices available locally")
        break
    except OSError as e:
        print(f"Drive connection dropped ({e}); reconnecting, attempt {attempt}/5 ...")
        time.sleep(5)
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        drive.mount('/content/drive', force_remount=True)
else:
    raise RuntimeError("Drive kept disconnecting -- restart the runtime and run this cell again")

## Train both variants (back to back)
Each variant trains until `EPOCHS` or `TIME_LIMIT_MIN`, whichever comes first, then
flushes Drive. If the runtime disconnects, re-run this cell -- finished runs are
skipped and the interrupted one resumes.

In [ ]:
ensure_drive()
for run in RUNS:
    ckpt_dir = f"{DRIVE_ROOT}/runs/{run['name']}"

    # Skip if already fully trained
    if os.path.exists(f"{ckpt_dir}/final.pt"):
        print(f"\n{'='*60}\n{run['name']}: already finished -- skipping\n{'='*60}")
        continue

    se_flag = "--use-se" if run["use_se"] else "--no-use-se"
    cmd = (f"cd {CODE_DIR} && python -u train.py --data-dir {LOCAL_DATA_DIR} "
           f"--checkpoint-dir {ckpt_dir} --af {AF} --kernel-t 3 "
           f"--sparse-domain {SPARSE_DOMAIN} --conv-mode {run['conv_mode']} {se_flag} "
           f"--epochs {EPOCHS} --batch-size {BATCH_SIZE} "
           f"--time-limit-min {TIME_LIMIT_MIN} --num-workers 2")
    print(f"\n{'='*60}\nTraining: {run['name']}\n  {cmd}\n{'='*60}")
    try:
        get_ipython().system(cmd)
    finally:
        flush_drive()

print("\nAll runs done (or use Runtime -> Run all again to continue interrupted ones).")

## Evaluate on the test split

In [ ]:
ensure_drive()
for run in RUNS:
    ckpt = f"{DRIVE_ROOT}/runs/{run['name']}/best.pt"
    if not os.path.exists(ckpt):
        print(f"{run['name']}: best.pt not found -- train first")
        continue
    print(f"\nEvaluating {run['name']} ...")
    get_ipython().system(f"cd {CODE_DIR} && python -u evaluate.py --data-dir {LOCAL_DATA_DIR} "
                         f"--checkpoint {ckpt} --split test")
flush_drive()

## Compare all AF 4 runs

In [ ]:
import glob, json
print(f"{'run':36s} {'split':5s} {'AF':<5s} {'method':13s} {'RLNE%':>7s}  {'PSNR':>6s}  {'SSIM%':>6s}")
print("-" * 90)
for path in sorted(glob.glob(f"{DRIVE_ROOT}/runs/dyn_af4_*/eval_*/results.json")):
    r = json.load(open(path))
    run = path.split("/runs/")[1].split("/")[0]
    for m, o in r["overall"].items():
        print(f"{run:36s} {r['split']:5s} AF{r['af_requested']:<4g} {m:13s} "
              f"{o['RLNE_mean'] * 100:6.2f}  {o['PSNR_mean']:6.2f}  {o['SSIM_mean'] * 100:6.2f}")

## Look at a reconstruction from the latest run

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for run in reversed(RUNS):
    ex_dir = f"{DRIVE_ROOT}/runs/{run['name']}/eval_test_af{AF:g}"
    if os.path.isdir(ex_dir):
        exs = sorted(glob.glob(f"{ex_dir}/*.npz"))
        if exs:
            ex = exs[0]
            d = np.load(ex)
            ref, zf, rec = (d[k].astype(float) for k in ("ref", "zero_filled", "odls_dynamic"))
            t, vmax = ref.shape[0] // 2, ref.max()
            fig, ax = plt.subplots(2, 3, figsize=(15, 10))
            for j, (img, title) in enumerate([(ref, "fully sampled"), (zf, f"zero-filled AF={AF}"),
                                               (rec, run["name"])]):
                ax[0, j].imshow(img[t], cmap="gray", vmin=0, vmax=vmax)
                ax[0, j].set_title(f"{title}, frame {t}")
                col = ref.shape[2] // 2
                ax[1, j].imshow(img[:, :, col].T, cmap="gray", vmin=0, vmax=vmax, aspect="auto")
                ax[1, j].set_title("y-t profile")
            for a in ax.ravel():
                a.axis("off")
            plt.suptitle(os.path.basename(ex))
            plt.tight_layout()
            plt.show()
            break
else:
    print("No evaluation results yet -- run the Evaluate cell first")

## Notes
- **Parameter counts** are printed at the start of each training run so you can verify
  the separable variant has fewer parameters.
- Re-running is safe: finished runs are skipped, interrupted runs resume.
- To also train at AF 8: duplicate this notebook, change `AF = 8`, and update `RUNS`
  names to `dyn_af8_kt3_sep_yt_v2` / `dyn_af8_kt3_sep_se_yt_v2`.
- The extended evaluation (`odls_eval_plus/`) can then be re-run with the new models
  added to its `MODELS` dict to get the full comparison table with Wilcoxon tests.